In [5]:
import pandas as pd
import numpy as np
import itertools
import time
from typing import List, Set, Dict, Tuple, FrozenSet, Any, Union

df = pd.read_csv('market.csv', sep=';')

print(f"Shape: {df.shape}")
df.head()

Shape: (464, 22)


,Bread,Honey,Bacon,Toothpaste,Banana,Apple,Hazelnut,Cheese,Meat,Carrot,...,Milk,Butter,ShavingFoam,Salt,Flour,HeavyCream,Egg,Olive,Shampoo,Sugar
0,1,0,1,0,1,1,1,0,0,1,...,0,0,0,0,0,1,1,0,0,1
1,1,1,1,0,1,1,1,0,0,0,...,1,1,0,0,1,0,0,1,1,0
2,0,1,1,1,1,1,1,1,1,0,...,1,0,1,1,1,1,1,0,0,1
3,1,1,0,1,0,1,0,0,0,0,...,1,0,0,0,1,0,1,1,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
transactions: List[FrozenSet[str]] = []

for index, row in df.iterrows():
    items = row[row == 1].index.tolist()
    if items:
        transactions.append(frozenset(items))


all_items = [item for t in transactions for item in t]
item_counts = pd.Series(all_items).value_counts()

print("\nTop 10 most frequent items (Corrected):")
print(item_counts.head(10))

total_trans = len(transactions)
if total_trans > 0:
    top_item_freq = item_counts.iloc[0]
    print(f"\nMost popular item support: {top_item_freq / total_trans:.4f} ({top_item_freq} times)")

# prog : 10%


Top 10 most frequent items (Corrected):
Banana         208
Cheese         206
Bacon          200
Hazelnut       195
HeavyCream     193
Honey          193
Carrot         192
Bread          189
ShavingFoam    188
Apple          188
Name: count, dtype: int64

Most popular item support: 0.4483 (208 times)
1% support count would be: 4.64


In [ ]:
def get_support(candidates: Set[FrozenSet], transactions: List[FrozenSet]) -> Dict[FrozenSet, int]:
    counts = {c: 0 for c in candidates}
    for t in transactions:
        for c in candidates:
            if c.issubset(t):
                counts[c] += 1
    return counts

def apriori_gen(freq_sets: List[FrozenSet], k: int) -> List[FrozenSet]:
    candidates = []
    # Join step: merge sets sharing k-2 items
    sorted_sets = sorted(list(freq_sets), key=lambda x: sorted(list(x)))
    n = len(sorted_sets)

    for i in range(n):
        for j in range(i + 1, n):
            l1, l2 = sorted(list(sorted_sets[i])), sorted(list(sorted_sets[j]))
            if l1[:k-2] == l2[:k-2]:
                cand = frozenset(l1) | frozenset(l2)
                # Prune step: all subsets must be frequent
                if all((cand - {x}) in freq_sets for x in cand):
                    candidates.append(cand)
    return candidates

def run_apriori(transactions: List[FrozenSet], min_sup: float) -> Dict[FrozenSet, float]:
    min_count = min_sup * len(transactions)
    # L1: Frequent 1-itemsets
    item_counts = pd.Series([i for t in transactions for i in t]).value_counts()
    curr_freq = [frozenset([i]) for i, c in item_counts.items() if c >= min_count]

    all_freq = {frozenset([i]): c/len(transactions) for i, c in item_counts.items() if c >= min_count}

    k = 2
    while curr_freq:
        cands = apriori_gen(set(curr_freq), k)
        if not cands: break

        cand_counts = get_support(cands, transactions)
        curr_freq = [c for c, count in cand_counts.items() if count >= min_count]

        for fs in curr_freq:
            all_freq[fs] = cand_counts[fs] / len(transactions)
        k += 1

    return all_freq

freq_itemsets = run_apriori(transactions, min_sup=0.1)
print(f"Found {len(freq_itemsets)} frequent itemsets.")

Found 549 frequent itemsets.


prog: 10%
join -> rozmiar k, laczylismy ze soba dwa zbiory o rozmiarze k-1, ale tylko takie, ktore mialy pierwsze k-2 elementy identyczne (zeby uniknac duplikatow)
prune -> dla kazdego kanddyata sprawdzalismy, czy kazdy jego podzbior byl wczenisje uznany za czesty. jesli jeden podzbior byl rzadki, caly kanddyuat ladowal w koszu. (nadzbior rzadkiego zbioru tez jest rzadki)
 